In [ ]:
import requests
import os
import zipfile
import pandas as pd
import glob
import duckdb
from tqdm import tqdm



In [ ]:
os.makedirs('dados_bolsa_familia', exist_ok=True)
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "pt-BR,pt;q=0.9",
    "Referer": "https://www.portaldatransparencia.gov.br/",
}

downloads = {
    2017: "https://www.portaltransparencia.gov.br/download-de-dados/bolsa-familia-pagamentos/201701",
    2018: "https://www.portaltransparencia.gov.br/download-de-dados/bolsa-familia-pagamentos/201801",
    2019: "https://www.portaltransparencia.gov.br/download-de-dados/bolsa-familia-pagamentos/201901",
    2020: "https://www.portaltransparencia.gov.br/download-de-dados/bolsa-familia-pagamentos/202001",
    2021: "https://www.portaltransparencia.gov.br/download-de-dados/bolsa-familia-pagamentos/202101",
    2023: "https://www.portaldatransparencia.gov.br/download-de-dados/novo-bolsa-familia/202303",
    2024: "https://www.portaldatransparencia.gov.br/download-de-dados/novo-bolsa-familia/202401",
    2025: "https://www.portaldatransparencia.gov.br/download-de-dados/novo-bolsa-familia/202501",
    2026: "https://www.portaldatransparencia.gov.br/download-de-dados/novo-bolsa-familia/202601",
}

for ano, url in downloads.items():
    print(f"Baixando {ano}...", end=" ")
    try:
        response = requests.get(url, headers=headers, timeout=120)
        if response.status_code == 200:
            zip_path = f"dados_bolsa_familia/{ano}.zip"
            with open(zip_path, 'wb') as f:
                f.write(response.content)
            with zipfile.ZipFile(zip_path, 'r') as z:
                z.extractall('dados_bolsa_familia/')
            os.remove(zip_path)
            print("OK")
        else:
            print(f"Erro HTTP {response.status_code}")
    except Exception as e:
        print(f"Falhou: {e}")

print("\nPronto! Arquivos em ./dados_bolsa_familia/")

In [4]:
import duckdb

con = duckdb.connect('dados_bolsa_familia/unificado.duckdb')

con.execute("DROP TABLE IF EXISTS bolsa_familia")

con.execute("""
    CREATE TABLE bolsa_familia AS
    SELECT * FROM read_csv(
        'dados_bolsa_familia/*.csv',
        delim=';',
        header=true,
        encoding='latin-1',
        ignore_errors=true
    )
""")

total = con.execute("SELECT COUNT(*) FROM bolsa_familia").fetchone()[0]
print(f"✅ Total de registros carregados: {total:,}")

con.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Total de registros carregados: 150,211,322


In [6]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None) 

con = duckdb.connect('dados_bolsa_familia/unificado.duckdb')

print("=== PRIMEIRAS LINHAS ===")
print(con.execute("SELECT * FROM bolsa_familia LIMIT 5").df().to_string())

print("\n=== COLUNAS E TIPOS ===")
print(con.execute("DESCRIBE bolsa_familia").df().to_string())

print("\n=== REGISTROS POR ANO ===")
print(con.execute("""
    SELECT 
        LEFT(CAST("MÊS COMPETÊNCIA" AS VARCHAR), 4) AS ano,
        COUNT(*) AS total_registros,
        COUNT(DISTINCT UF) AS total_ufs
    FROM bolsa_familia
    GROUP BY ano
    ORDER BY ano
""").df().to_string())

con.close()

=== PRIMEIRAS LINHAS ===
   MÊS COMPETÊNCIA  MÊS REFERÊNCIA  UF CÓDIGO MUNICÍPIO SIAFI     NOME MUNICÍPIO  CPF FAVORECIDO  NIS FAVORECIDO                 NOME FAVORECIDO VALOR PARCELA
0           201701          201601  MA                   0723            BACABAL  ***.673.533-**     16550200718   TEREZINHA DE FATIMA ESPINDOLA         46,00
1           201701          201601  SP                   6639            LIMEIRA  ***.695.168-**     16081867537     ANGELA MARIA MESSIAS MORAES         46,00
2           201701          201602  AC                   0109        MANCIO LIMA  ***.667.452-**     16379762982  MARIA DEILDES BARBOSA MESQUITA        163,00
3           201701          201602  AC                   0151  PLACIDO DE CASTRO  ***.647.802-**     16601338657       VAL JOSE ALMIRO DE MATTOS        202,00
4           201701          201602  AC                   0139         RIO BRANCO  ***.030.952-**     20453008245      MARIA DAMIANA SILVA ARAUJO         46,00

=== COLUNAS E TIPOS 